<a href="https://colab.research.google.com/github/alexpanayi474/DSC511-MovieLens-Project/blob/main/TMDB_Enrichment_for_MovieLens2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>



# TMDB Enrichment for MovieLens

This notebook enriches MovieLens movies with selected TMDB metadata. It saves checkpoint CSV files during the run so progress is not lost if Colab disconnects.

In [ ]:
# Mount Google Drive so the MovieLens files and checkpoint files are available.
#from google.colab import drive
# drive.mount('/content/gdrive')

# Project data folder in Google Drive.
google_drive_path = "/content/gdrive/MyDrive/Colab Notebooks/Data/DSC_511_PROJECT/"
google_drive_path = "G:/My Drive/Colab Notebooks/Data/DSC_511_PROJECT/"  # for local testing

In [2]:
# Core libraries for data handling, API requests, parallel execution, and checkpoints.
import os
import csv
import time
import threading
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import pandas as pd
import requests

In [ ]:
# TMDB API key.
# Keep the key in one place so it is easy to replace or hide before submission.
TMDB_API_KEY = "55d206bac9cdb44a8069f01785a51884"

# Performance settings.
# SAMPLE_START / SAMPLE_SIZE control which batch to run in a single Colab session.
# Set SAMPLE_START=0 and SAMPLE_SIZE=None to process all movies in one go
# (may hit the Colab runtime limit for very large datasets).
# Alternatively, run batches of 10 000:
#   batch 1 → SAMPLE_START=0,     SAMPLE_SIZE=10000
#   batch 2 → SAMPLE_START=10000, SAMPLE_SIZE=10000  … and so on.
SAMPLE_START = 0
SAMPLE_SIZE = None  #10000  # set to None to process ALL movies at once
MAX_WORKERS = 20
SAVE_EVERY = 250

# Output files saved in Google Drive.
OUTPUT_DIR = Path(google_drive_path) / "enrichment_cache"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CHECKPOINT_FILE = OUTPUT_DIR / "tmdb_metadata_selected.csv"
ENRICHED_MOVIES_FILE = OUTPUT_DIR / "movies_enriched_selected.csv"

print("Checkpoint file:", CHECKPOINT_FILE)
print("Enriched output:", ENRICHED_MOVIES_FILE)


Checkpoint file: G:\My Drive\Colab Notebooks\Data\DSC_511_PROJECT\enrichment_cache\tmdb_metadata_selected.csv
Enriched output: G:\My Drive\Colab Notebooks\Data\DSC_511_PROJECT\enrichment_cache\movies_enriched_selected.csv


In [3]:
# Load MovieLens files.
# links.csv connects MovieLens movieId to TMDB and IMDb identifiers.
ratings_df = pd.read_csv(Path(google_drive_path) / "ratings.csv", usecols=["movieId"])
movies_df = pd.read_csv(Path(google_drive_path) / "movies.csv")
links_df = pd.read_csv(Path(google_drive_path) / "links.csv")

print("ratings rows:", len(ratings_df))
print("movies rows:", len(movies_df))
print("links rows:", len(links_df))

ratings rows: 33832162
movies rows: 86537
links rows: 86537


In [ ]:
# Build the list of movies to enrich from movies_df (all movies, not just rated ones).
# links_df provides the tmdbId mapping; movies without a valid tmdbId are kept
# in the final CSV but enriched columns will be empty (NaN).
links_with_tmdb = links_df[
    links_df["tmdbId"].notna()
].copy()

links_with_tmdb["movieId"] = links_with_tmdb["movieId"].astype(int)
links_with_tmdb["tmdbId"]  = links_with_tmdb["tmdbId"].astype(int)

# Merge movies_df with links to get the tmdbId per movie.
# We keep ALL movies (how='left') so movies without a tmdbId (e.g. movieId=126)
# are still present in the scrape list — they simply have no tmdbId and will be
# skipped by the API fetch, then retained as empty rows in the final CSV.
movies_with_links = movies_df.merge(links_with_tmdb[["movieId", "tmdbId", "imdbId"]],
                                    on="movieId", how="left")

# Restrict to movies that have a valid tmdbId for the API scrape.
links_filtered = movies_with_links[movies_with_links["tmdbId"].notna()].copy()
links_filtered["tmdbId"] = links_filtered["tmdbId"].astype(int)

# Apply optional batch window (change SAMPLE_START/SAMPLE_SIZE above).
if SAMPLE_SIZE is not None:
    links_filtered = links_filtered.iloc[SAMPLE_START:SAMPLE_START + SAMPLE_SIZE]

print("Total movies in movies.csv:", len(movies_df))
print("Movies with a valid tmdbId:", len(movies_with_links[movies_with_links['tmdbId'].notna()]))
print("Movies selected for this TMDB enrichment run:", len(links_filtered))
links_filtered.head()


Total movies in movies.csv: 86537
Movies with a valid tmdbId: 86411
Movies selected for this TMDB enrichment run: 86411


,movieId,title,genres,tmdbId,imdbId
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,862,114709.0
1,2,Jumanji (1995),Adventure|Children|Fantasy,8844,113497.0
2,3,Grumpier Old Men (1995),Comedy|Romance,15602,113228.0
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,31357,114885.0
4,5,Father of the Bride Part II (1995),Comedy,11862,113041.0


In [ ]:
# Selected output fields.
# Only the columns needed for the project are fetched from TMDB.
FIELDNAMES = [
    "movieId",
    "tmdb_id",
    "budget",
    "revenue",
    "runtime",
    "release_date",
    "original_language",
    "director",
    "top_cast",
    "production_companies",
    "production_countries",
    "keywords"
]


In [ ]:
import pandas as pd
import csv
from pathlib import Path

# Assuming FIELDNAMES and CHECKPOINT_FILE are defined in previous cells or globally accessible
# Load previous checkpoint results if the file already exists.
# Rerunning the notebook continues from the missing movies only.
def load_checkpoint(path):
    if not path.exists():
        return pd.DataFrame(columns=FIELDNAMES), set()

    saved_df = pd.read_csv(path, low_memory=False)
    # Use pd.to_numeric with errors='coerce' to handle non-numeric values gracefully
    # Then drop NaNs and convert to integer type
    completed_ids = set(pd.to_numeric(saved_df["tmdb_id"], errors='coerce').dropna().astype(int))
    return saved_df, completed_ids


# Append rows to the checkpoint file without rewriting the whole file every time.
def append_checkpoint(path, rows):
    if not rows:
        return

    write_header = not path.exists() or path.stat().st_size == 0
    with open(path, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=FIELDNAMES, extrasaction="ignore")
        if write_header:
            writer.writeheader()
        writer.writerows(rows)

saved_df, completed_tmdb_ids = load_checkpoint(CHECKPOINT_FILE)
print("Previously saved movies:", len(completed_tmdb_ids))

Previously saved movies: 85204


In [ ]:
# One HTTP session per worker thread improves speed by reusing connections.
thread_local = threading.local()

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/124.0 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9",
}


def get_session():
    if not hasattr(thread_local, "session"):
        session = requests.Session()
        session.headers.update(HEADERS)
        thread_local.session = session
    return thread_local.session

In [ ]:
# Fetch selected TMDB metadata for one movie.
# append_to_response=credits,keywords brings director, cast, and keywords in the same request.
def fetch_tmdb_metadata(row, retries=3):
    movie_id = int(row["movieId"])
    tmdb_id  = int(row["tmdbId"])

    url = f"https://api.themoviedb.org/3/movie/{tmdb_id}"
    params = {
        "api_key": TMDB_API_KEY,
        "append_to_response": "credits,keywords",
    }

    for attempt in range(retries):
        try:
            response = get_session().get(url, params=params, timeout=20)

            # Missing TMDB pages are skipped gracefully.
            if response.status_code == 404:
                return None

            # Rate limits are retried with a short wait.
            if response.status_code == 429:
                time.sleep(2 + attempt * 3)
                continue

            response.raise_for_status()
            data = response.json()

            crew = data.get("credits", {}).get("crew", [])
            cast = data.get("credits", {}).get("cast", [])

            directors = [p.get("name") for p in crew if p.get("job") == "Director"]

            release_date = data.get("release_date") or None

            # budget=0 is treated as missing by TMDB; store as None.
            raw_budget = data.get("budget")
            budget = raw_budget if raw_budget else None

            return {
                "movieId":           movie_id,
                "tmdb_id":           tmdb_id,
                "budget":            budget,
                "revenue":           data.get("revenue") or None,
                "runtime":           data.get("runtime") or None,
                "release_date":      release_date,
                "original_language": data.get("original_language"),
                "director":          "|".join(directors[:3]) if directors else None,
                # top 3 cast members
                "top_cast":          "|".join(p.get("name", "") for p in cast[:3]) if cast else None,
                "production_companies": "|".join(c.get("name", "") for c in data.get("production_companies", [])) or None,
                "production_countries": "|".join(c.get("name", "") for c in data.get("production_countries", [])) or None,
                "keywords":          "|".join(k.get("name", "") for k in data.get("keywords", {}).get("keywords", [])) or None,

            }

        except requests.RequestException:
            time.sleep(1 + attempt * 2)

    return None


In [ ]:
# Build the pending list by removing movies already saved in the checkpoint.
pending_df = links_filtered[~links_filtered["tmdbId"].isin(completed_tmdb_ids)].copy()

print("Movies in current run:", len(links_filtered))
print("Already completed:", len(completed_tmdb_ids))
print("Pending movies:", len(pending_df))

Movies in current run: 86411
Already completed: 85204
Pending movies: 1171


In [ ]:
pip install aiohttp

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import aiohttp
import asyncio

SEMAPHORE_LIMIT = 30  # max concurrent requests

async def fetch_tmdb_async(session, row, semaphore, retries=3):
    movie_id = int(row["movieId"])
    tmdb_id = int(row["tmdbId"])
    url = f"https://api.themoviedb.org/3/movie/{tmdb_id}"
    params = {"api_key": TMDB_API_KEY, "append_to_response": "credits,keywords"}

    for attempt in range(retries):
        try:
            async with semaphore:
                async with session.get(url, params=params, timeout=aiohttp.ClientTimeout(total=20)) as resp:
                    if resp.status == 404:
                        return None
                    if resp.status == 429:
                        await asyncio.sleep(2 + attempt * 3)
                        continue
                    resp.raise_for_status()
                    data = await resp.json()

            crew = data.get("credits", {}).get("crew", [])
            cast = data.get("credits", {}).get("cast", [])
            directors = [p.get("name") for p in crew if p.get("job") == "Director"]
            raw_budget = data.get("budget")

            return {
                "movieId": movie_id,
                "tmdb_id": tmdb_id,
                "budget": raw_budget if raw_budget else None,
                "revenue": data.get("revenue") or None,
                "runtime": data.get("runtime") or None,
                "release_date": data.get("release_date") or None,
                "original_language": data.get("original_language"),
                "director": "|".join(directors[:3]) if directors else None,
                "top_cast": "|".join(p.get("name", "") for p in cast[:3]) if cast else None,
                "production_companies": "|".join(c.get("name", "") for c in data.get("production_companies", [])) or None,
                "production_countries": "|".join(c.get("name", "") for c in data.get("production_countries", [])) or None,
                "keywords": "|".join(k.get("name", "") for k in data.get("keywords", {}).get("keywords", [])) or None,
            }
        except Exception:
            await asyncio.sleep(1 + attempt * 2)
    return None


async def run_scrape(pending_rows):
    semaphore = asyncio.Semaphore(SEMAPHORE_LIMIT)
    buffer = []
    success = 0
    failed = 0

    async with aiohttp.ClientSession(headers=HEADERS) as session:
        tasks = [fetch_tmdb_async(session, row, semaphore) for row in pending_rows]

        for i, coro in enumerate(asyncio.as_completed(tasks), 1):
            result = await coro
            if result:
                buffer.append(result)
                success += 1
            else:
                failed += 1

            if len(buffer) >= SAVE_EVERY:
                append_checkpoint(CHECKPOINT_FILE, buffer)
                print(f"Checkpoint | done={i:,} success={success:,} failed={failed:,}")
                buffer = []

            if i % 2000 == 0:
                print(f"Progress {i:,}/{len(pending_rows):,}")

    append_checkpoint(CHECKPOINT_FILE, buffer)
    print(f"Done. Success: {success:,} | Failed: {failed:,}")

# Run it
rows = pending_df.to_dict("records")
await run_scrape(rows)

Done. Success: 0 | Failed: 1,171


In [ ]:
'''
# Run TMDB requests in parallel.
# This loop continues through the whole selected subset and saves progress every SAVE_EVERY successful rows.
rows = pending_df.to_dict("records")
buffer = []
success_count = 0
failed_count = 0
start_time = time.time()

print(f"Starting TMDB enrichment for {len(rows):,} pending movies...")

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = [executor.submit(fetch_tmdb_metadata, row) for row in rows]

    for completed, future in enumerate(as_completed(futures), start=1):
        result = future.result()

        if result is None:
            failed_count += 1
        else:
            buffer.append(result)
            success_count += 1

        # Checkpoint saves partial progress before the full scrape completes.
        if len(buffer) >= SAVE_EVERY:
            append_checkpoint(CHECKPOINT_FILE, buffer)
            buffer = []
            elapsed_min = (time.time() - start_time) / 60
            print(f"Saved checkpoint | completed={completed:,} success={success_count:,} failed={failed_count:,} elapsed={elapsed_min:.1f} min")

        # Progress display for long runs.
        if completed % 500 == 0 or completed == len(rows):
            elapsed_min = (time.time() - start_time) / 60
            print(f"Progress {completed:,}/{len(rows):,} | success={success_count:,} failed={failed_count:,} elapsed={elapsed_min:.1f} min")

# Final checkpoint save for the remaining rows.
append_checkpoint(CHECKPOINT_FILE, buffer)

print("TMDB enrichment finished.")
print("Successful new rows:", success_count)
print("Failed or skipped rows:", failed_count)
'''

'\n# Run TMDB requests in parallel.\n# This loop continues through the whole selected subset and saves progress every SAVE_EVERY successful rows.\nrows = pending_df.to_dict("records")\nbuffer = []\nsuccess_count = 0\nfailed_count = 0\nstart_time = time.time()\n\nprint(f"Starting TMDB enrichment for {len(rows):,} pending movies...")\n\nwith ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:\n    futures = [executor.submit(fetch_tmdb_metadata, row) for row in rows]\n\n    for completed, future in enumerate(as_completed(futures), start=1):\n        result = future.result()\n\n        if result is None:\n            failed_count += 1\n        else:\n            buffer.append(result)\n            success_count += 1\n\n        # Checkpoint saves partial progress before the full scrape completes.\n        if len(buffer) >= SAVE_EVERY:\n            append_checkpoint(CHECKPOINT_FILE, buffer)\n            buffer = []\n            elapsed_min = (time.time() - start_time) / 60\n            p

In [ ]:
# Load the full checkpoint after scraping.
# Duplicate tmdb_id rows are removed in case a run was restarted near a checkpoint.
tmdb_metadata_df = pd.read_csv(CHECKPOINT_FILE)
tmdb_metadata_df = tmdb_metadata_df.drop_duplicates(subset=["tmdb_id"], keep="last")

# Re-save the cleaned checkpoint.
tmdb_metadata_df.to_csv(CHECKPOINT_FILE, index=False)

print("Total saved TMDB rows:", len(tmdb_metadata_df))
tmdb_metadata_df.head()

Total saved TMDB rows: 85204


,movieId,tmdb_id,budget,revenue,runtime,release_date,original_language,director,top_cast,production_companies,production_countries,keywords
0,1,862,30000000.0,401157969.0,81.0,1995-11-22,en,John Lasseter,Tom Hanks|Tim Allen|Don Rickles,Pixar,United States of America,rescue|friendship|mission|jealousy|villain|bul...
1,30,37557,2300000.0,NaN,108.0,1995-09-14,zh,Zhang Yimou,Gong Li|Li Baotian|Sun Chun,Alpha Films|La Sept Cinéma|UGC|Shanghai Film S...,China|France,"shanghai, china|chinese mafia|coming of age|mi..."
2,11,9087,62000000.0,107879496.0,113.0,1995-11-17,en,Rob Reiner,Michael Douglas|Annette Bening|Martin Sheen,Castle Rock Entertainment|Universal Pictures|W...,United States of America,new love|usa president|the white house|holiday...
3,10,710,60000000.0,352194034.0,130.0,1995-11-16,en,Martin Campbell,Pierce Brosnan|Sean Bean|Izabella Scorupco,EON Productions,United Kingdom,computer virus|cuba|falsely accused|secret int...
4,3,15602,25000000.0,71518503.0,101.0,1995-12-22,en,Howard Deutch,Walter Matthau|Jack Lemmon|Ann-Margret,Lancaster Gate|Warner Bros. Pictures,United States of America,fishing|sequel|old man|best friend|wedding|ita...


In [ ]:
# Merge TMDB metadata back into movies_df using a left join on movieId.
# This guarantees ALL movies from movies.csv appear in the output.
# Movies without a tmdbId (e.g. movieId=126) will have NaN in enrichment columns.
movies_enriched_df = movies_df.merge(
    tmdb_metadata_df[["movieId"] + [c for c in FIELDNAMES if c != "movieId"]],
    on="movieId",
    how="left"
)

movies_enriched_df.to_csv(ENRICHED_MOVIES_FILE, index=False)

print("Saved enriched movies file:", ENRICHED_MOVIES_FILE)
print("Total rows (should equal movies.csv):", len(movies_enriched_df))
print("Movies WITHOUT tmdb data (no tmdbId or 404):",
      movies_enriched_df["tmdb_id"].isna().sum())
movies_enriched_df.head()


Saved enriched movies file: G:\My Drive\Colab Notebooks\Data\DSC_511_PROJECT\enrichment_cache\movies_enriched_selected.csv
Total rows (should equal movies.csv): 86537
Movies WITHOUT tmdb data (no tmdbId or 404): 1333


,movieId,title,genres,tmdb_id,budget,revenue,runtime,release_date,original_language,director,top_cast,production_companies,production_countries,keywords
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,862.0,30000000.0,401157969.0,81.0,1995-11-22,en,John Lasseter,Tom Hanks|Tim Allen|Don Rickles,Pixar,United States of America,rescue|friendship|mission|jealousy|villain|bul...
1,2,Jumanji (1995),Adventure|Children|Fantasy,8844.0,65000000.0,262821940.0,104.0,1995-12-15,en,Joe Johnston,Robin Williams|Kirsten Dunst|Bradley Pierce,TriStar Pictures|Interscope Communications|Tei...,United States of America,giant insect|board game|disappearance|jungle|r...
2,3,Grumpier Old Men (1995),Comedy|Romance,15602.0,25000000.0,71518503.0,101.0,1995-12-22,en,Howard Deutch,Walter Matthau|Jack Lemmon|Ann-Margret,Lancaster Gate|Warner Bros. Pictures,United States of America,fishing|sequel|old man|best friend|wedding|ita...
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,31357.0,16000000.0,81452156.0,127.0,1995-12-22,en,Forest Whitaker,Whitney Houston|Angela Bassett|Loretta Devine,20th Century Fox,United States of America,based on novel or book|single mother|divorce|a...
4,5,Father of the Bride Part II (1995),Comedy,11862.0,NaN,76594107.0,106.0,1995-12-08,en,Charles Shyer,Steve Martin|Diane Keaton|Martin Short,Touchstone Pictures|Sandollar Productions,United States of America,daughter|baby|parent child relationship|midlif...


In [ ]:
# Data quality summary for the enriched metadata columns.
enrichment_cols = [c for c in FIELDNAMES if c in movies_enriched_df.columns]
print(movies_enriched_df[enrichment_cols].isna().sum().sort_values(ascending=False))


revenue                 70705
budget                  70470
keywords                24932
production_companies    11512
production_countries     5573
top_cast                 4464
runtime                  2018
director                 1834
release_date             1367
tmdb_id                  1333
original_language        1333
movieId                     0
dtype: int64


In [ ]:
movies_enriched_df[movies_enriched_df["tmdb_id"].isna()]

,movieId,title,genres,tmdb_id,budget,revenue,runtime,release_date,original_language,director,top_cast,production_companies,production_countries,keywords
596,604,Criminals (1996),Documentary,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
706,721,Halfmoon (Paul Bowles - Halbmond) (1995),Drama,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
715,730,Low Life (1994),Drama,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
754,770,Costa Brava (1946),Drama,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
775,791,"Last Klezmer: Leopold Kozlowski, His Life and ...",Documentary,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
85888,286851,Werewolf Women of the S.S. (2007),Comedy|Horror,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
86093,287439,Muppet*Vision 3-D (1991),(no genres listed),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
86282,288069,American Masters: Miles Davis - Birth of the Cool,Documentary,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
86365,288447,A House A Little Too Quiet,(no genres listed),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
check_cols = ["top_cast", "runtime", "director", "release_date"]

has_tmdb = movies_enriched_df[movies_enriched_df["tmdb_id"].notna()]
has_any_missing = has_tmdb[check_cols].isna().any(axis=1).sum()

print(f"Movies with valid tmdb_id: {len(has_tmdb):,}")
print(f"  All 4 fields complete: {len(has_tmdb) - has_any_missing:,}")
print(f"  At least one missing: {has_any_missing:,}")
print()
print(has_tmdb[check_cols].isna().sum())

Movies with valid tmdb_id: 85,204
  All 4 fields complete: 81,175
  At least one missing: 4,029

top_cast        3131
runtime          685
director         501
release_date      34
dtype: int64


In [ ]:
# ── Release date mismatch analysis ──
# Only movies with a valid tmdb_id (exclude 1,333 without TMDB match)
df = movies_enriched_df[movies_enriched_df["tmdb_id"].notna()].copy()

df["tmdb_year"] = pd.to_datetime(df["release_date"], errors="coerce").dt.year
df["title_year"] = df["title"].str.extract(r"\((\d{4})\)$").astype(float)

# Group 1: year mismatch (both exist but differ)
mismatch_mask = (df["tmdb_year"].notna() & df["title_year"].notna() &
                 (df["tmdb_year"] != df["title_year"]))

# Group 2: no year in title (have release_date but can't verify)
no_title_year_mask = df["release_date"].notna() & df["title_year"].isna()

# Group 3: no release_date at all
no_date_mask = df["release_date"].isna()

# Group 4: matching — all good
match_mask = (df["tmdb_year"].notna() & df["title_year"].notna() &
              (df["tmdb_year"] == df["title_year"]))

print(f"Movies with valid tmdb_id: {len(df):,}")
print(f"   Year matches:          {match_mask.sum():,}")
print(f"   Year mismatch:         {mismatch_mask.sum():,}")
print(f"   No year in title:      {no_title_year_mask.sum():,}")
print(f"   No release date:       {no_date_mask.sum():,}")
print()

# We will fetch IMDb dates for groups 1 + 2 + 3
combined_mask = mismatch_mask | no_title_year_mask | no_date_mask

fix_df = df[combined_mask][["movieId"]].merge(
    links_df[["movieId", "imdbId"]], on="movieId"
)
fix_df = fix_df[fix_df["imdbId"].notna()].copy()
fix_df["imdbId"] = fix_df["imdbId"].astype(int)

print(f"Total to fetch from IMDb: {len(fix_df):,}")

Movies with valid tmdb_id: 85,204
   Year matches:          78,870
   Year mismatch:         5,640
   No year in title:      660
   No release date:       34

Total to fetch from IMDb: 6,334


In [ ]:
# ── Release date mismatch analysis ──
# Only movies with valid tmdb_id (exclude 1,333 without TMDB match)
df = movies_enriched_df[movies_enriched_df["tmdb_id"].notna()].copy()

df["tmdb_year"] = pd.to_datetime(df["release_date"], errors="coerce").dt.year
df["title_year"] = df["title"].str.extract(r"\((\d{4})\)$").astype(float)

has_both = df["tmdb_year"].notna() & df["title_year"].notna()
match_mask = has_both & (df["tmdb_year"] == df["title_year"])
mismatch_mask = has_both & (df["tmdb_year"] != df["title_year"])
no_title_year = df["title_year"].isna()
no_date_has_title = df["release_date"].isna() & df["title_year"].notna()
no_date_no_title = df["release_date"].isna() & df["title_year"].isna()

print(f"Movies with valid tmdb_id: {len(df):,}")
print(f"   Year matches:                    {match_mask.sum():,}")
print(f"   Year mismatch:                   {mismatch_mask.sum():,}  → will fetch from IMDb")
print(f"   No year in title (total):         {no_title_year.sum():,}  → unverifiable, will set to null")
print(f"   No release date + has title year: {no_date_has_title.sum():,}  → will fetch from IMDb")
print(f"   No release date + no title year:  {no_date_no_title.sum():,}  → unverifiable, will set to null")

Movies with valid tmdb_id: 85,204
   Year matches:                    78,870
   Year mismatch:                   5,640  → will fetch from IMDb
   No year in title (total):         691  → unverifiable, will set to null
   No release date + has title year: 3  → will fetch from IMDb
   No release date + no title year:  31  → unverifiable, will set to null


## Fix Mismatched Release Dates via IMDb (OMDb API)

The TMDB `release_date` often reflects the US theatrical release, which can differ from
the original release year shown in the MovieLens title. We identified movies where the
TMDB year does not match the title year, and movies with no release date but a verifiable
title year.

We fetch release dates from IMDb (via the OMDb API) for two groups:
1. **Year mismatch** — TMDB year ≠ MovieLens title year
2. **No release date + has title year** — can verify the IMDb result against the title

We do NOT fetch for movies with no year in the title (~660), as we cannot verify any
source. These will be set to null. Movies with no release date AND no title year will
also remain null.

In [ ]:
# ── Identify movies to fetch from IMDb ──
fetch_mask = mismatch_mask | no_date_has_title

fix_df = df[fetch_mask][["movieId"]].merge(
    links_df[["movieId", "imdbId"]], on="movieId"
)
fix_df = fix_df[fix_df["imdbId"].notna()].copy()
fix_df["imdbId"] = fix_df["imdbId"].astype(int)

print(f"Movies to fetch from IMDb: {len(fix_df):,}")

Movies to fetch from IMDb: 5,643


In [ ]:
# ── Fetch release dates from OMDb (IMDb data) ──
import aiohttp
import asyncio

OMDB_API_KEY = "19ead468"

async def fetch_omdb_date(session, movie_id, imdb_id, semaphore, retries=3):
    imdb_str = f"tt{imdb_id:07d}"
    url = f"http://www.omdbapi.com/?i={imdb_str}&apikey={OMDB_API_KEY}"

    for attempt in range(retries):
        try:
            async with semaphore:
                async with session.get(url, timeout=aiohttp.ClientTimeout(total=15)) as resp:
                    data = await resp.json(content_type=None)
                    released = data.get("Released")
                    if released and released != "N/A":
                        dt = pd.to_datetime(released, format="%d %b %Y", errors="coerce")
                        if pd.notna(dt):
                            return {"movieId": movie_id, "imdb_release_date": dt.strftime("%Y-%m-%d")}
                    return None
        except Exception:
            await asyncio.sleep(1 + attempt * 2)
    return None


async def run_omdb_scrape(rows):
    semaphore = asyncio.Semaphore(10)
    results = []

    async with aiohttp.ClientSession() as session:
        tasks = [fetch_omdb_date(session, r["movieId"], r["imdbId"], semaphore)
                 for r in rows]

        for i, coro in enumerate(asyncio.as_completed(tasks), 1):
            result = await coro
            if result:
                results.append(result)
            if i % 500 == 0:
                print(f"Progress: {i:,}/{len(rows):,} | found: {len(results):,}")

    print(f"Done. Retrieved {len(results):,} dates from IMDb.")
    return pd.DataFrame(results)


rows = fix_df.to_dict("records")
imdb_dates_df = await run_omdb_scrape(rows)

Progress: 500/5,643 | found: 467
Progress: 1,000/5,643 | found: 946
Progress: 1,500/5,643 | found: 1,417
Progress: 2,000/5,643 | found: 1,881
Progress: 2,500/5,643 | found: 2,358
Progress: 3,000/5,643 | found: 2,827
Progress: 3,500/5,643 | found: 3,298
Progress: 4,000/5,643 | found: 3,765
Progress: 4,500/5,643 | found: 4,237
Progress: 5,000/5,643 | found: 4,723
Progress: 5,500/5,643 | found: 5,203
Done. Retrieved 5,337 dates from IMDb.


In [ ]:
# ── Apply IMDb dates ──
if len(imdb_dates_df) > 0:
    date_map = imdb_dates_df.set_index("movieId")["imdb_release_date"]
    mask = movies_enriched_df["movieId"].isin(date_map.index)
    movies_enriched_df.loc[mask, "release_date"] = (
        movies_enriched_df.loc[mask, "movieId"].map(date_map)
    )
    print(f"Updated {mask.sum():,} release dates from IMDb")

fetched_ids = set(imdb_dates_df["movieId"]) if len(imdb_dates_df) > 0 else set()
tried_ids = set(fix_df["movieId"])
not_found_ids = tried_ids - fetched_ids

# ── Overall verification ──
df_verify = movies_enriched_df[movies_enriched_df["movieId"].isin(tried_ids)].copy()
df_verify["_year"] = pd.to_datetime(df_verify["release_date"], errors="coerce").dt.year
df_verify["_title_year"] = df_verify["title"].str.extract(r"\((\d{4})\)$").astype(float)

has_date = df_verify["release_date"].notna()
year_matches = has_date & (df_verify["_year"] == df_verify["_title_year"])
year_mismatches = has_date & df_verify["_title_year"].notna() & (df_verify["_year"] != df_verify["_title_year"])

print(f"\n── Overall recovery for {len(tried_ids):,} movies we tried to fix ──")
print(f"  IMDb date found:         {len(fetched_ids):,}")
print(f"    Year matches title:    {year_matches.sum():,}  → fixed ")
print(f"    Year still mismatches: {year_mismatches.sum():,}  → will be set to null")
print(f"  IMDb date not found:     {len(not_found_ids):,}  → will remain null")

# ── Breakdown: originally no release date + had title year ──
no_date_ids = set(df[no_date_has_title]["movieId"])
recovered = no_date_ids & fetched_ids
still_missing = no_date_ids - fetched_ids

if recovered:
    rec_df = movies_enriched_df[movies_enriched_df["movieId"].isin(recovered)].copy()
    rec_df["_year"] = pd.to_datetime(rec_df["release_date"], errors="coerce").dt.year
    rec_df["_title_year"] = rec_df["title"].str.extract(r"\((\d{4})\)$").astype(float)
    rec_match = (rec_df["_year"] == rec_df["_title_year"]).sum()
    rec_mismatch = (rec_df["_year"] != rec_df["_title_year"]).sum()
else:
    rec_match = rec_mismatch = 0

print(f"\n── Subset: had NO release date + had title year ({len(no_date_ids)}) ──")
print(f"  Recovered from IMDb:     {len(recovered)}")
print(f"  Still no date:           {len(still_missing)}  → will remain null")

Updated 5,337 release dates from IMDb

── Overall recovery for 5,643 movies we tried to fix ──
  IMDb date found:         5,337
    Year matches title:    1,626  → fixed 
    Year still mismatches: 4,014  → will be set to null
  IMDb date not found:     306  → will remain null

── Subset: had NO release date + had title year (3) ──
  Recovered from IMDb:     0
  Still no date:           3  → will remain null


## Final Release Date Cleanup

After applying IMDb dates, we still have movies with unreliable release dates:
- Movies where both TMDB and IMDb disagree with the MovieLens title year
- Movies where OMDb returned no result
- Movies with no year in their title (~660), making verification impossible

Since we cannot reliably determine the correct release date for any of these,
we set them all to null.

In [ ]:
# ── Set all unreliable release dates to null ──
df_clean = movies_enriched_df.copy()
df_clean["_year"] = pd.to_datetime(df_clean["release_date"], errors="coerce").dt.year
df_clean["_title_year"] = df_clean["title"].str.extract(r"\((\d{4})\)$").astype(float)

# 1. Still mismatched after IMDb fix
still_mismatch = (df_clean["_year"].notna() & df_clean["_title_year"].notna() &
                  (df_clean["_year"] != df_clean["_title_year"]))

# 2. No year in title — unverifiable
no_title_year_mask = df_clean["release_date"].notna() & df_clean["_title_year"].isna()

# 3. Movies we tried to fix but OMDb had no result
not_found_mask = df_clean["movieId"].isin(not_found_ids)

null_mask = still_mismatch | no_title_year_mask | not_found_mask

movies_enriched_df.loc[null_mask, "release_date"] = None

print(f"Set to null:")
print(f"  Still mismatched:     {still_mismatch.sum():,}")
print(f"  No year in title:     {no_title_year_mask.sum():,}")
print(f"  OMDb not found:       {not_found_mask.sum():,}")
print(f"  Total set to null:    {null_mask.sum():,} (some overlap)")
print(f"\nTotal missing release_date now (including 1333 with no tmdbid): {movies_enriched_df['release_date'].isna().sum():,}")

Set to null:
  Still mismatched:     0
  No year in title:     0
  OMDb not found:       306
  Total set to null:    306 (some overlap)

Total missing release_date now (including 1333 with no tmdbid): 6,041


In [ ]:
# ── Final data quality summary ──
has_tmdb = movies_enriched_df[movies_enriched_df["tmdb_id"].notna()].copy()

check_cols = ["release_date", "runtime", "director", "top_cast"]

print(f"=== Missing values per column (excl. {movies_enriched_df['tmdb_id'].isna().sum():,} without tmdb_id) ===")
print(has_tmdb[check_cols].isna().sum().sort_values(ascending=False))
print()

has_any_missing = has_tmdb[check_cols].isna().any(axis=1).sum()
all_complete = len(has_tmdb) - has_any_missing

print(f"Movies with valid tmdb_id: {len(has_tmdb):,}")
print(f"  All 4 fields complete:   {all_complete:,}")
print(f"  At least one missing:    {has_any_missing:,}")

=== Missing values per column (excl. 1,333 without tmdb_id) ===
release_date    4708
top_cast        3131
runtime          685
director         501
dtype: int64

Movies with valid tmdb_id: 85,204
  All 4 fields complete:   76,797
  At least one missing:    8,407


In [ ]:
movies_enriched_df.to_csv(ENRICHED_MOVIES_FILE, index=False)
print(f"Saved: {ENRICHED_MOVIES_FILE}")

Saved: G:\My Drive\Colab Notebooks\Data\DSC_511_PROJECT\enrichment_cache\movies_enriched_selected.csv


## Validate release dates against first rating timestamps

This reloads the final enriched CSV saved above, reloads MovieLens ratings with timestamps, and flags movies where the earliest rating happened before the scraped release date. Those rows are useful review candidates because MovieLens ratings should normally appear on or after a movie's release date.


In [7]:
# Project data folder in Google Drive.
google_drive_path = "/content/gdrive/MyDrive/Colab Notebooks/Data/DSC_511_PROJECT/"
project_data_path = Path(google_drive_path)

enriched_path = project_data_path / "movies_enriched_selected.csv"
ratings_path = project_data_path / "ratings.csv"

movies_final_df = pd.read_csv(enriched_path, low_memory=False)
ratings_with_time_df = pd.read_csv(ratings_path, usecols=["movieId", "timestamp"])

print("Loaded enriched movies:", len(movies_final_df), "from", enriched_path)
print("Loaded ratings:", len(ratings_with_time_df), "from", ratings_path)


Loaded enriched movies: 86537 from /content/gdrive/MyDrive/Colab Notebooks/Data/DSC_511_PROJECT/movies_enriched_selected.csv
Loaded ratings: 33832162 from /content/gdrive/MyDrive/Colab Notebooks/Data/DSC_511_PROJECT/ratings.csv


In [5]:
# Compare each movie's release date with its earliest MovieLens rating timestamp.
first_rating_df = (
    ratings_with_time_df
    .groupby("movieId", as_index=False)["timestamp"]
    .min()
    .rename(columns={"timestamp": "first_rating_timestamp"})
)

first_rating_df["first_rating_datetime"] = (
    pd.to_datetime(first_rating_df["first_rating_timestamp"], unit="s", utc=True)
    .dt.tz_convert(None)
)
first_rating_df["first_rating_date"] = first_rating_df["first_rating_datetime"].dt.normalize()

release_check_df = movies_final_df.merge(first_rating_df, on="movieId", how="left")
release_check_df["release_datetime"] = pd.to_datetime(
    release_check_df["release_date"], errors="coerce"
).dt.normalize()

release_check_df["days_from_release_to_first_rating"] = (
    release_check_df["first_rating_date"] - release_check_df["release_datetime"]
).dt.days

release_check_df["title_year"] = pd.to_numeric(
    release_check_df["title"].str.extract(r"\((\d{4})\)\s*$")[0], errors="coerce"
)
release_check_df["release_year"] = release_check_df["release_datetime"].dt.year
release_check_df["title_year_mismatch"] = (
    release_check_df["release_year"].notna()
    & release_check_df["title_year"].notna()
    & (release_check_df["release_year"] != release_check_df["title_year"])
)

# Main red flag: a rating timestamp before the scraped release date.
release_check_df["rating_before_release"] = release_check_df["days_from_release_to_first_rating"] < 0

review_cols = [
    "movieId", "title", "release_date", "first_rating_datetime",
    "days_from_release_to_first_rating", "title_year", "release_year",
    "title_year_mismatch", "rating_before_release", "tmdb_id"
]

release_date_review_df = release_check_df.loc[
    release_check_df["rating_before_release"] | release_check_df["title_year_mismatch"],
    review_cols
].sort_values(["rating_before_release", "days_from_release_to_first_rating"], ascending=[False, True])

print("Movies with a parsed release date:", release_check_df["release_datetime"].notna().sum())
print("Movies with at least one rating timestamp:", release_check_df["first_rating_datetime"].notna().sum())
print("Ratings before scraped release date:", release_check_df["rating_before_release"].sum())
print("Release year mismatches title year:", release_check_df["title_year_mismatch"].sum())

release_date_review_df.head(30)


Movies with a parsed release date: 80496
Movies with at least one rating timestamp: 83239
Ratings before scraped release date: 2157
Release year mismatches title year: 0


,movieId,title,release_date,first_rating_datetime,days_from_release_to_first_rating,title_year,release_year,title_year_mismatch,rating_before_release,tmdb_id
25120,122914,Avengers: Infinity War - Part II (2019),2019-04-24,2015-03-25 12:46:42,-1491.0,2019.0,2019.0,False,True,299534.0
30252,135426,Fantastic Beasts and Where to Find Them 2 (2018),2018-11-14,2015-06-20 15:34:36,-1243.0,2018.0,2018.0,False,True,338952.0
33794,143347,Aquaman (2018),2018-12-07,2015-11-13 20:21:57,-1120.0,2018.0,2018.0,False,True,297802.0
25119,122912,Avengers: Infinity War - Part I (2018),2018-04-25,2015-06-05 18:01:11,-1055.0,2018.0,2018.0,False,True,299536.0
25121,122916,Thor: Ragnarok (2017),2017-10-02,2015-01-23 17:52:15,-983.0,2017.0,2017.0,False,True,284053.0
25112,122898,Justice League (2017),2017-11-15,2015-06-08 08:18:17,-891.0,2017.0,2017.0,False,True,141052.0
25111,122896,Pirates of the Caribbean: Dead Men Tell No Tal...,2017-05-23,2015-01-21 21:04:16,-853.0,2017.0,2017.0,False,True,166426.0
33808,143387,Pitch Perfect 3 (2017),2017-12-20,2015-10-02 11:27:41,-810.0,2017.0,2017.0,False,True,353616.0
867,887,Talk of Angels (1998),1998-10-30,1996-09-17 11:17:43,-773.0,1998.0,1998.0,False,True,56077.0
25122,122918,Guardians of the Galaxy 2 (2017),2017-04-19,2015-06-08 08:18:27,-681.0,2017.0,2017.0,False,True,283995.0


In [6]:
# Save the review table next to the enriched CSV for manual inspection.
review_output_path = enriched_path.parent / "release_date_rating_timestamp_check.csv"
release_date_review_df.to_csv(review_output_path, index=False)
print("Saved release-date review candidates:", review_output_path)

# Show the strongest timestamp-based red flags first.
release_date_review_df[release_date_review_df["rating_before_release"]].head(50)


Saved release-date review candidates: /content/gdrive/MyDrive/Colab Notebooks/Data/DSC_511_PROJECT/release_date_rating_timestamp_check.csv


,movieId,title,release_date,first_rating_datetime,days_from_release_to_first_rating,title_year,release_year,title_year_mismatch,rating_before_release,tmdb_id
25120,122914,Avengers: Infinity War - Part II (2019),2019-04-24,2015-03-25 12:46:42,-1491.0,2019.0,2019.0,False,True,299534.0
30252,135426,Fantastic Beasts and Where to Find Them 2 (2018),2018-11-14,2015-06-20 15:34:36,-1243.0,2018.0,2018.0,False,True,338952.0
33794,143347,Aquaman (2018),2018-12-07,2015-11-13 20:21:57,-1120.0,2018.0,2018.0,False,True,297802.0
25119,122912,Avengers: Infinity War - Part I (2018),2018-04-25,2015-06-05 18:01:11,-1055.0,2018.0,2018.0,False,True,299536.0
25121,122916,Thor: Ragnarok (2017),2017-10-02,2015-01-23 17:52:15,-983.0,2017.0,2017.0,False,True,284053.0
25112,122898,Justice League (2017),2017-11-15,2015-06-08 08:18:17,-891.0,2017.0,2017.0,False,True,141052.0
25111,122896,Pirates of the Caribbean: Dead Men Tell No Tal...,2017-05-23,2015-01-21 21:04:16,-853.0,2017.0,2017.0,False,True,166426.0
33808,143387,Pitch Perfect 3 (2017),2017-12-20,2015-10-02 11:27:41,-810.0,2017.0,2017.0,False,True,353616.0
867,887,Talk of Angels (1998),1998-10-30,1996-09-17 11:17:43,-773.0,1998.0,1998.0,False,True,56077.0
25122,122918,Guardians of the Galaxy 2 (2017),2017-04-19,2015-06-08 08:18:27,-681.0,2017.0,2017.0,False,True,283995.0
